# Setup

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import WORKLOADS_DIR, TRINO_CATALOG
from lakehouse.render import render_collective
from lakehouse.lakehouse import Lakehouse

namespace = ""
workload_name = "DiverSQL"
instance_name = "lakehouse-a"
schema = "tpcds"

# Deploy Lakehouse

In [ ]:
"""
Deploy lakehouse with given YAML and register to HMS
"""

res = render_collective(
    templates_dir=str(ROOT / "lakehouse" / "manifests"),
    values_path=str(ROOT / "lakehouse" / "lakehouses" / "lakehouse-tpcds.yaml"),
)

lh = Lakehouse(res, verbose=False)

lh.deploy()
lh.register_schema("tpcds", "/mnt/iceberg/warehouse", use_dot_db=True) 

# Generate Workloads

In [ ]:
"""
Generate a workload
"""

lh = Lakehouse.from_release(instance_name=instance_name, namespace=namespace,sync_schemas=True, verbose=True)

report = lh.generate_workload(
    schema=schema,
    workload_name=workload_name,
    num_queries=100,
    min_tables=2,
    max_tables=16,
    warmup=True,
    validate=True,
    batch_size=100,
    generation_workers=64,
    validation_workers=8,
    reasoning="high",
)

In [ ]:
"""
Generate a single SQL query
"""

lh = Lakehouse.from_release(instance_name=instance_name, namespace=namespace, sync_schemas=False, verbose=False)

generated_query = lh.generate_query(schema=schema, return_metadata=True)

print(generated_query["sql"])
print(generated_query["tables_used"])

# Analyse Queries/Workloads Plans

In [ ]:
"""
Analyse query workload
"""

lh = Lakehouse.from_release(instance_name=instance_name, namespace=namespace, sync_schemas=True, verbose=False)

plan_bundle = lh.load_query_plans(
    workload_path=str(WORKLOADS_DIR / workload_name),
    schema=schema,
)

out = lh.workload_diversity_metrics(
    plan_bundle,
    catalog=TRINO_CATALOG,
    schema=schema,
)

# Execute Queries/Workloads

In [ ]:
"""
Issue a single query to the lakehouse
"""
lh = Lakehouse.from_release(instance_name=instance_name, namespace=namespace, sync_schemas=True, verbose=False)

try:
    query = generated_query["sql"]
except NameError:
    query = """
            Select 1
            """
    
resullt = lh.issue_query(
    query,
    schema=schema,
    receive_result=True,
    fetch_ui_doc=False,
    query_plan=False,
    io_plan=False,
)

print(f"{resullt}")


In [ ]:
"""
Run a workload on the lakehouse
"""
NUM_ATTEMPTS = 5

lh = Lakehouse.from_release(instance_name=instance_name, namespace=namespace, sync_schemas=False, verbose=False)

for i in range(NUM_ATTEMPTS):
    res = lh.run_workload(
        workload_name=workload_name,
        schema=schema,
        attempts=1, # Cache realted
        query=True,
        query_plan = True,
        io_plan = True,
    )

# Tear Down Lakehouse

In [ ]:
"""
Tear down lakehouse
"""

lh = Lakehouse.from_release(instance_name=instance_name, namespace=namespace,sync_schemas=False, verbose=False)

lh.tear_down()